In [ ]:
%pip install datasets
%pip install datasets groq python-dotenv ragas
%pip install "langchain-community==0.4.1"
%pip install --upgrade ragas instructor groq
%pip install sentence-transformers

Evaluator LLM initialized successfully!


In [8]:

import os
from dotenv import load_dotenv
from openai import OpenAI
from ragas.llms import llm_factory

load_dotenv()

# Groq API through OpenAI-compatible client
groq_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

# Ragas evaluator
evaluator_llm = llm_factory(
    model="openai/gpt-oss-120b",
    provider="openai",
    client=groq_client,
    adapter="instructor"
)

print("Evaluator LLM initialized successfully!")


Evaluator LLM initialized successfully!


In [9]:

import os
import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI

from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextRecall,
    ContextPrecision
)

# --------------------------------------------------
# 1. Load Groq API Key
# --------------------------------------------------

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file")

# --------------------------------------------------
# 2. Initialize Async Groq Client
# --------------------------------------------------

groq_client = AsyncOpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1",
    timeout=120.0,
    max_retries=3
)

# --------------------------------------------------
# 3. Initialize Ragas Evaluator LLM
# --------------------------------------------------

evaluator_llm = llm_factory(
    model="openai/gpt-oss-120b",
    provider="openai",
    client=groq_client,
    adapter="instructor"
)

print("Groq evaluator initialized!")

# --------------------------------------------------
# 4. Initialize Hugging Face Embeddings
# --------------------------------------------------

evaluator_embeddings = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

print("Hugging Face embeddings initialized!")

# --------------------------------------------------
# 5. Evaluation Dataset
# --------------------------------------------------

eval_data = {
    "question": [
        "Within how many days of purchase can customers "
        "request a full software refund if unsatisfied?"
    ],
    "contexts": [[
        "Customers who are unsatisfied with the software "
        "can request a full refund within 14 days of purchase."
    ]],
    "answer": [
        "Customers who are unsatisfied with the software "
        "can request a full refund within 14 days of purchase."
    ],
    "ground_truth": [
        "Customers who are unsatisfied with the software "
        "can request a full refund within 14 days of purchase."
    ]
}

# --------------------------------------------------
# 6. Initialize All Evaluation Metrics
# --------------------------------------------------

faithfulness_metric = Faithfulness(
    llm=evaluator_llm
)

answer_relevancy_metric = AnswerRelevancy(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

context_recall_metric = ContextRecall(
    llm=evaluator_llm
)

context_precision_metric = ContextPrecision(
    llm=evaluator_llm
)

# --------------------------------------------------
# 7. Run Asynchronous RAG Evaluation
# --------------------------------------------------

results = []

for question, contexts, answer, reference in zip(
    eval_data["question"],
    eval_data["contexts"],
    eval_data["answer"],
    eval_data["ground_truth"]
):

    # Generation: Faithfulness
    faithfulness_result = await faithfulness_metric.ascore(
        user_input=question,
        response=answer,
        retrieved_contexts=contexts
    )

    # Generation: Answer Relevancy
    relevancy_result = await answer_relevancy_metric.ascore(
        user_input=question,
        response=answer
    )

    # Retrieval: Context Recall
    recall_result = await context_recall_metric.ascore(
        user_input=question,
        retrieved_contexts=contexts,
        reference=reference
    )

    # Retrieval: Context Precision
    precision_result = await context_precision_metric.ascore(
        user_input=question,
        retrieved_contexts=contexts,
        reference=reference
    )

    # Generation: Derived Hallucination Rate
    faithfulness_score = float(faithfulness_result.value)

    hallucination_rate = (
        1.0 - faithfulness_score
        if pd.notna(faithfulness_score)
        else float("nan")
    )

    results.append({
        "question": question,
        "faithfulness": faithfulness_score,
        "answer_relevancy": relevancy_result.value,
        "context_recall": recall_result.value,
        "context_precision": precision_result.value,
        "hallucination_rate": hallucination_rate
    })

# --------------------------------------------------
# 8. Display Evaluation Results
# --------------------------------------------------

results_df = pd.DataFrame(results)

print("\nRAG Evaluation Results:")
print(results_df.to_string(index=False))

# --------------------------------------------------
# 9. Average Metric Scores
# --------------------------------------------------

print("\nAverage Scores:")

print(
    results_df[
        [
            "faithfulness",
            "answer_relevancy",
            "context_recall",
            "context_precision",
            "hallucination_rate"
        ]
    ].mean().round(4)
)

# --------------------------------------------------
# 10. Save Results
# --------------------------------------------------

# results_df.to_csv(
#     "rag_evaluation_results.csv",
#     index=False
# )


Groq evaluator initialized!


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9797.55it/s]


Hugging Face embeddings initialized!

RAG Evaluation Results:
                                                                                     question  faithfulness  answer_relevancy  context_recall  context_precision  hallucination_rate
Within how many days of purchase can customers request a full software refund if unsatisfied?           1.0          0.866229             1.0                1.0                 0.0

Average Scores:
faithfulness          1.0000
answer_relevancy      0.8662
context_recall        1.0000
context_precision     1.0000
hallucination_rate    0.0000
dtype: float64
